# Customer Churn Risk Ranking: Quickstart

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xuefeiyu2015/customer-churn-ml/blob/main/notebooks/quickstart.ipynb)

This notebook runs the **existing `customer_churn_ml` package** end to end. It does not reimplement any of it:

1. Repository setup and dependency installation (uv, locked versions)
2. Dataset download with checksum verification
3. Training with the package CLI (`python -m customer_churn_ml.training`)
4. Evaluation metrics
5. Loading the saved model
6. Predictions from raw customer features

**How it runs:** the project requires Python ≥ 3.13. The notebook kernel (e.g. Colab's Python) only handles orchestration and display using the standard library and pandas. All package code runs through `uv run`, which uses the project's own locked Python 3.13 environment. The holdout split is the same stratified 80/20 split with `random_state=42` used throughout the project.

*Static benchmark: scores rank churn risk in a historical snapshot. They are uncalibrated ranking scores, not validated churn probabilities or 30-day forecasts, and they do not measure retention-campaign impact.*

## 1. Repository setup and dependencies

In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

REPO_URL = os.environ.get("CHURN_REPO_URL", "https://github.com/xuefeiyu2015/customer-churn-ml.git")
BRANCH = os.environ.get("CHURN_BRANCH", "main")


def find_repo_root(start: Path):
    for path in [start, *start.parents]:
        if (path / "pyproject.toml").exists() and (path / "src" / "customer_churn_ml").is_dir():
            return path
    return None


def run(cmd):
    """Run a command, show its output, and stop the notebook if it fails."""
    print("$", " ".join(str(c) for c in cmd))
    proc = subprocess.run([str(c) for c in cmd], capture_output=True, text=True)
    print(proc.stdout[-4000:], proc.stderr[-4000:], sep="\n")
    proc.check_returncode()


# uv should always target the project's own .venv, not a caller's activated environment
os.environ.pop("VIRTUAL_ENV", None)

ROOT = find_repo_root(Path.cwd())
if ROOT is None:  # fresh runtime such as Colab: clone the repository
    run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, "customer-churn-ml"])
    ROOT = Path("customer-churn-ml").resolve()
os.chdir(ROOT)
print("Repository root:", ROOT)
print("Notebook kernel Python:", sys.version.split()[0])

In [ ]:
if shutil.which("uv") is None:
    # Pinned to the uv release that produced uv.lock (inside the pyproject's uv_build range).
    run([sys.executable, "-m", "pip", "install", "-q", "uv==0.11.16"])

# Locked install; uv downloads Python 3.13 if the runtime does not have it.
# --inexact keeps extra packages already present (e.g. a local Jupyter kernel).
run(["uv", "sync", "--locked", "--inexact"])
run(["uv", "run", "--locked", "python", "--version"])

## 2. Dataset

IBM Telco Customer Churn (7,043 customers, 21 columns). CSV files are not committed, so the file is downloaded from the public mlbookcamp mirror and checked against a known SHA-256 hash.

In [ ]:
import hashlib
import urllib.request

DATA_URL = ("https://raw.githubusercontent.com/alexeygrigorev/mlbookcamp-code/master/"
            "chapter-03-churn-prediction/WA_Fn-UseC_-Telco-Customer-Churn.csv")
DATA_SHA256 = "88be4b93fbe0cc83421af1c503794c97c342eca914c1576db7c276e61d61358a"
DATA_PATH = ROOT / "data" / "WA_Fn-UseC_-Telco-Customer-Churn.csv"

if not DATA_PATH.exists():
    DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
    urllib.request.urlretrieve(DATA_URL, DATA_PATH)

digest = hashlib.sha256(DATA_PATH.read_bytes()).hexdigest()
assert digest == DATA_SHA256, f"Unexpected dataset checksum: {digest}"
print("Dataset OK:", DATA_PATH, f"({DATA_PATH.stat().st_size:,} bytes)")

## 3. Train with the package CLI

This fits the full Pipeline (cleaning → encoding/scaling → RandomForest) on the training split, evaluates the holdout once, computes out-of-fold metrics, saves the `.joblib` and `.json` files, then reloads the model and checks that its predictions are identical. Artifacts are written to a temporary directory, so the repository stays clean.

In [ ]:
import tempfile

MODEL_PATH = Path(tempfile.gettempdir()) / "churn-quickstart" / "churn_random_forest.joblib"
os.environ["MODEL_PATH"] = str(MODEL_PATH)
os.environ["DATA_PATH"] = str(DATA_PATH)

run(["uv", "run", "--locked", "python", "-m", "customer_churn_ml.training",
     "--data", DATA_PATH, "--model-out", MODEL_PATH, "--overwrite"])

## 4. Evaluation metrics

Read from the metadata file written next to the model. K is scaled from 500 contacts per 7,043 customers: K = 100 on the 1,409-customer holdout and K = 400 for out-of-fold predictions on the 5,634 training customers. OOF AP is computed once over all pooled out-of-fold predictions, so it differs slightly from the mean of per-fold AP values.

In [ ]:
import json

import pandas as pd

meta = json.loads(MODEL_PATH.with_suffix(".json").read_text())["metadata"]
rows = ["average_precision", "roc_auc", "precision_at_k", "lift_at_k", "k", "n", "prevalence"]
metrics = pd.DataFrame({"holdout": meta["holdout_metrics"], "out-of-fold (train)": meta["oof_metrics"]}).loc[rows]
metrics = metrics.astype(object).map(lambda v: f"{v:,.0f}" if float(v).is_integer() else f"{v:.4f}")
print("Model:", meta["model"], meta["model_params"])
print("Dataset SHA-256:", meta["data"]["sha256"])
metrics

## 5–6. Load the saved model and score raw customers

This runs inside the project environment with `uv run`. The loaded Pipeline accepts raw rows exactly as they appear in the CSV (original column names, `Yes`/`No` strings, `TotalCharges` as text). No preprocessing is needed in the notebook. The output is an **uncalibrated risk score for ranking**, not a validated churn probability. Only load joblib files from trusted sources.

In [ ]:
%%bash
uv run --locked python - <<'EOF'
import os

import pandas as pd

from customer_churn_ml.evaluation import positive_class_proba
from customer_churn_ml.persistence import load_model

model = load_model(os.environ["MODEL_PATH"])  # complete fitted Pipeline
print([name for name, _ in model.steps])

# (a) existing customers, raw CSV rows; customerID is ignored, Churn is not needed
customers = pd.read_csv(os.environ["DATA_PATH"]).drop(columns="Churn")
risk = positive_class_proba(model, customers)  # uncalibrated risk scores, used for ranking
ranked = customers.assign(risk_score=risk.round(3)).sort_values("risk_score", ascending=False)
cols = ["customerID", "tenure", "Contract", "InternetService", "MonthlyCharges", "risk_score"]
print("\nTop 5 of", len(ranked), "customers by risk score:")
print(ranked[cols].head().to_string(index=False))

# (b) a brand-new customer written by hand, in raw form (blank TotalCharges, tenure 0)
new_customer = pd.DataFrame([{
    "gender": "Female", "SeniorCitizen": 0, "Partner": "No", "Dependents": "No",
    "tenure": 0, "PhoneService": "Yes", "MultipleLines": "No",
    "InternetService": "Fiber optic", "OnlineSecurity": "No", "OnlineBackup": "No",
    "DeviceProtection": "No", "TechSupport": "No", "StreamingTV": "Yes",
    "StreamingMovies": "Yes", "Contract": "Month-to-month", "PaperlessBilling": "Yes",
    "PaymentMethod": "Electronic check", "MonthlyCharges": 95.5, "TotalCharges": " ",
}])
score = float(positive_class_proba(model, new_customer)[0])
print("\nNew customer risk score (uncalibrated):", round(score, 3))
EOF

## Next steps

- Full analysis and model selection: [`notebooks/churn_analysis.ipynb`](churn_analysis.ipynb)
- Package code: `src/customer_churn_ml/`; tests: `uv run pytest -q`